In [1]:
!pip install --quiet minatar imageio-ffmpeg
!pip install tensorboard

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.5/29.5 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 150.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 21.3 MB/s eta 0:00:00


In [2]:
import os
DRIVE = "/content/drive/MyDrive/rl-final-project"
os.makedirs(os.path.join(DRIVE, "dqn_cnn"), exist_ok=True)
os.makedirs(os.path.join(DRIVE, "visuals", "dqn_cnn_rollout"), exist_ok=True)
os.makedirs(os.path.join(DRIVE, "plots", "dqn_cnn"), exist_ok=True)

print("Drive dir:", DRIVE)

Drive dir: /content/drive/MyDrive/rl-final-project


In [3]:
# Cell 2 — imports & device
import random, time, pickle
from collections import deque, namedtuple
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
import matplotlib.pyplot as plt
from PIL import Image
import imageio
import minatar

# Tensorboard logging
from torch.utils.tensorboard import SummaryWriter

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [4]:
# Cell 3 — wrapper: MiniAtariStack (4 frames)
class MiniAtariStack:
    def __init__(self, game_name="breakout", stack=4):
        self.env = minatar.Environment(game_name)
        self.stack = stack
        self.n_actions = self.env.num_actions()
        ss = self.env.state_shape()
        if isinstance(ss, int):
            self.h = self.w = int(np.sqrt(ss))
        elif len(ss) == 1:
            self.h = self.w = int(np.sqrt(ss[0]))
        else:
            self.h, self.w = ss[0], ss[1]
        self.frames = deque(maxlen=stack)

    def reset(self):
        self.env.reset()
        st = self.env.state()
        f = self._single_preproc(st)
        self.frames.clear()
        for _ in range(self.stack):
            self.frames.append(f.copy())
        return self._get_state()

    def step(self, action):
        r, done = self.env.act(int(action))
        st = self.env.state()
        f = self._single_preproc(st)
        self.frames.append(f.copy())
        return self._get_state(), float(r), bool(done)

    def _single_preproc(self, st):
        arr = np.array(st, dtype=np.float32)
        if arr.ndim == 3:
            arr = arr.sum(axis=2)           # collapse channels -> single plane
        mn, mx = float(arr.min()), float(arr.max())
        rng = mx - mn if mx > mn else 1.0
        arr = (arr - mn) / rng
        return arr

    def _get_state(self):
        return np.stack(list(self.frames), axis=0).astype(np.float32)  # (C,H,W)

    def render_raw(self):
        raw = self.frames[-1]
        img = (raw * 255).astype(np.uint8)
        return img

In [5]:
# Cell 4 — ReplayBuffer
Transition = namedtuple('Transition', ('state','action','reward','next_state','done'))

class ReplayBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append(Transition(state.copy(), int(action), float(reward), next_state.copy(), float(done)))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        return Transition(*zip(*batch))

    def __len__(self):
        return len(self.buffer)

In [6]:
# Cell 5 — DQNCNN
class DQNCNN(nn.Module):
    def __init__(self, in_channels=4, n_actions=6):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, stride=1, padding=0),  # 10x10 -> 8x8
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=0),           # 8x8 -> 6x6
            nn.ReLU()
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*6*6, 512),
            nn.ReLU(),
            nn.Linear(512, n_actions)
        )

    def forward(self, x):
        return self.fc(self.conv(x))

In [7]:
# Cell 6 — utils
def linear_eps(step, eps_start=1.0, eps_final=0.05, eps_decay=150000):
    if step >= eps_decay: return eps_final
    return eps_final + (eps_start - eps_final) * (1 - step/eps_decay)

def save_pickle(obj, path):
    with open(path, 'wb') as f:
        pickle.dump(obj, f)

In [8]:
# Cell 7 — train_dqn_cnn_improved (Double DQN)
def train_dqn_cnn_improved(env,
                           drive_dir,
                           num_steps=250000,
                           buffer_capacity=200000,
                           batch_size=32,
                           gamma=0.99,
                           lr=2.5e-4,
                           target_update_freq=5000,
                           start_learning=10000,
                           eps_start=1.0,
                           eps_final=0.05,
                           eps_decay=150000,
                           save_every=10000,
                           eval_every=20000,
                           use_double=True,
                           grad_clip=10.0,
                           log_dir=None):
    # prepare paths and writer
    out_dir = os.path.join(drive_dir, "dqn_cnn")
    os.makedirs(out_dir, exist_ok=True)
    model_path = os.path.join(out_dir, "best_dqn_cnn.pth")
    expert_dataset_path = os.path.join(out_dir, "expert_dataset_cnn.pkl")
    rewards_plot = os.path.join(drive_dir, "plots", "dqn_cnn", "dqn_cnn_rewards.png")

    writer = SummaryWriter(log_dir) if log_dir else None

    # models
    n_actions = env.n_actions
    net = DQNCNN(in_channels=env.stack, n_actions=n_actions).to(device)
    target = DQNCNN(in_channels=env.stack, n_actions=n_actions).to(device)
    target.load_state_dict(net.state_dict())
    opt = optim.Adam(net.parameters(), lr=lr)

    replay = ReplayBuffer(buffer_capacity)

    all_steps = 0
    episode_reward = 0.0
    episode_rewards = []
    expert_transitions = []
    best_avg = -float('inf')
    losses = []

    state = env.reset()

    start_time = time.time()
    last_save_time = start_time

    while all_steps < num_steps:
        eps = linear_eps(all_steps, eps_start, eps_final, eps_decay)
        # action selection (epsilon-greedy)
        if random.random() < eps:
            action = random.randrange(n_actions)
        else:
            with torch.no_grad():
                s = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
                qvals = net(s)
                action = int(torch.argmax(qvals, dim=1).item())

        # step
        next_state, reward, done = env.step(action)

        # store
        replay.push(state, action, reward, next_state, done)
        expert_transitions.append((state.copy(), int(action), float(reward), next_state.copy(), float(done)))

        episode_reward += reward
        all_steps += 1
        state = next_state

        # artificial logging episode chunk
        if all_steps % 1000 == 0:
            episode_rewards.append(episode_reward)
            if writer:
                writer.add_scalar("train/episode_reward_chunk", episode_reward, all_steps)
            episode_reward = 0.0
            state = env.reset()

        # learning step
        if len(replay) > start_learning:
            batch = replay.sample(batch_size)
            states = np.stack(batch.state).astype(np.float32)    # (B,C,H,W)
            next_states = np.stack(batch.next_state).astype(np.float32)
            actions = np.array(batch.action, dtype=np.int64)
            rewards = np.array(batch.reward, dtype=np.float32)
            dones = np.array(batch.done, dtype=np.float32)

            states_t = torch.tensor(states, dtype=torch.float32).to(device)
            next_states_t = torch.tensor(next_states, dtype=torch.float32).to(device)
            actions_t = torch.tensor(actions, dtype=torch.long).unsqueeze(1).to(device)
            rewards_t = torch.tensor(rewards, dtype=torch.float32).unsqueeze(1).to(device)
            dones_t = torch.tensor(dones, dtype=torch.float32).unsqueeze(1).to(device)

            # current Q
            q_values = net(states_t).gather(1, actions_t)  # (B,1)

            # Double DQN logic:
            with torch.no_grad():
                if use_double:
                    # actions from online network
                    next_actions = torch.argmax(net(next_states_t), dim=1, keepdim=True)  # (B,1)
                    q_next_target = target(next_states_t).gather(1, next_actions)         # use target values for chosen actions
                else:
                    q_next_target = target(next_states_t).max(1)[0].unsqueeze(1)

                q_target_val = rewards_t + gamma * (1 - dones_t) * q_next_target

            loss = nn.functional.mse_loss(q_values, q_target_val)
            opt.zero_grad()
            loss.backward()
            # gradient clipping
            torch.nn.utils.clip_grad_norm_(net.parameters(), grad_clip)
            opt.step()
            losses.append(loss.item())
            if writer:
                writer.add_scalar("train/loss", loss.item(), all_steps)

        # update target network periodically
        if all_steps % target_update_freq == 0:
            target.load_state_dict(net.state_dict())

        # logging & save
        if all_steps % save_every == 0:
            avg_recent = float(np.mean(episode_rewards[-50:])) if episode_rewards else 0.0
            print(f"[{time.strftime('%H:%M:%S')}] Step {all_steps}/{num_steps} eps={eps:.3f} avg_recent={avg_recent:.3f} replay_len={len(replay)}")
            if writer:
                writer.add_scalar("train/eps", eps, all_steps)
                writer.add_scalar("train/avg_recent", avg_recent, all_steps)
            if avg_recent > best_avg:
                best_avg = avg_recent
                torch.save(net.state_dict(), model_path)
                print("Saved improved model ->", model_path)

        # periodic eval (optional hook)
        if all_steps % eval_every == 0:
            pass

    # final save & cleanup
    torch.save(net.state_dict(), model_path)
    save_pickle(expert_transitions, expert_dataset_path)
    print("TRAIN COMPLETE. model:", model_path, "dataset:", expert_dataset_path)
    if writer:
        writer.close()

    # save rewards plot
    if episode_rewards:
        plt.figure(figsize=(10,4))
        plt.plot(episode_rewards)
        plt.title("DQN-CNN episode rewards (chunked)")
        plt.grid(True)
        plt.savefig(rewards_plot, dpi=150)
        plt.close()

    return {"model_path": model_path, "expert_dataset_path": expert_dataset_path, "rewards_plot": rewards_plot}

In [9]:
# Cell 8 — run: debug или финал
env = MiniAtariStack(game_name="breakout", stack=4)

# Debug run (быстро проверить работоспособность)
# res = train_dqn_cnn_improved(env, DRIVE, num_steps=20000, buffer_capacity=50000, batch_size=32, lr=2.5e-4, start_learning=2000, target_update_freq=1000, save_every=5000, eps_decay=15000, log_dir=os.path.join(DRIVE,'logs','dqn_debug'))

# Final recommended run (долго — запускай, если есть время/ресурсы)
res = train_dqn_cnn_improved(
    env,
    DRIVE,
    num_steps=200000,           # для хорошего результата; можно 400k
    buffer_capacity=200000,
    batch_size=32,
    lr=2.5e-4,
    start_learning=10000,
    target_update_freq=5000,
    save_every=20000,
    eps_decay=150000,
    log_dir=os.path.join(DRIVE, "logs", "dqn_cnn_final"),
    use_double=True
)
print(res)

[11:49:04] Step 20000/200000 eps=0.873 avg_recent=0.700 replay_len=20000
Saved improved model -> /content/drive/MyDrive/rl-final-project/dqn_cnn/best_dqn_cnn.pth


KeyboardInterrupt: 

In [ ]:
# Cell 9 — evaluate & gif
def evaluate_policy(env, net, episodes=5, steps=500, save_folder=None):
    net.eval()
    scores = []
    if save_folder:
        os.makedirs(save_folder, exist_ok=True)
    for ep in range(episodes):
        s = env.reset()
        total = 0.0
        frames = []
        for t in range(steps):
            with torch.no_grad():
                x = torch.tensor(s, dtype=torch.float32).unsqueeze(0).to(device)
                q = net(x)
                a = int(torch.argmax(q, dim=1).item())
            s, r, done = env.step(a)
            total += r
            raw = env.render_raw()
            up = Image.fromarray((raw).astype(np.uint8)).resize((200,200), Image.NEAREST)
            frames.append(np.array(up))
            if done:
                break
        scores.append(total)
        if save_folder:
            imageio.mimsave(os.path.join(save_folder, f"eval_ep{ep:02d}.gif"), frames, fps=12)
    return np.array(scores)

# Usage (после того как модель обучена):
# net = DQNCNN(in_channels=4, n_actions=env.n_actions).to(device)
# net.load_state_dict(torch.load(res['model_path'], map_location=device))
# eval_scores = evaluate_policy(env, net, episodes=5, steps=600, save_folder=os.path.join(DRIVE,'visuals','dqn_cnn_rollout'))
# print("Eval mean:", eval_scores.mean(), "std:", eval_scores.std())